# Exercise: Prompt Management with Amazon Bedrock

The exercises in this course will have an associated charge in your AWS account. In this exercise, you will create the following resources:

- **Amazon Bedrock**

The final task in this exercise includes instructions to delete all the resources that you create.

Familiarize yourself with [Amazon Bedrock pricing](https://aws.amazon.com/bedrock/pricing/) and the [AWS Free Tier](https://aws.amazon.com/free/).

## Exercise Overview

In this exercise, you'll use Amazon Bedrock APIs to:

✅ Create reusable prompt templates using the `create_prompt` API

✅ Invoke foundation models with structured input using the `converse` API

✅ Optimize prompt performance with the `optimize_prompt` API

✅ Version and iterate on prompts

✅ Delete prompts from Prompt Management

💼 **The use case for this exercise** is to utilize generative AI to generate professional, compelling job descriptions from structured hiring criteria. This mimics how HR teams or recruiting platforms can use AI to ensure consistent, high-quality messaging for job openings.

## Prerequisites

Before you start, you need to have:

- Python 3.x
- AWS CLI configured with AWS credentials (`aws configure`). You can use the AWS exerciseuser credentials set up in exercise 1 in the Getting Started with AWS Generative AI for Developers course.
- Installed dependencies and created a new virtual environment.

To create a new Python environment and install the needed packages for the upcoming steps, run the command that corresponds to your operating system:

**macOS:**
```bash
cd ~/Desktop && python3 -m venv exercise4 && source exercise4/bin/activate && pip install boto3 jupyter
```

**Windows:**
```powershell
cd ~/Desktop; python -m venv exercise4; .\exercise4\Scripts\Activate.ps1; pip install boto3 jupyter
```

## Troubleshooting

🤔 If you get stuck or run into any errors, go back a few steps to ensure you didn't miss any instructions.

🛠️ Still having trouble? Try these steps:

🔁 Rerun the previous cells to ensure your environment is correctly set up.

🧹 Restart your kernel and clear output to reset the environment.

📋 Double-check for typos in code or parameter names.

🌐 Make sure your internet connection is active (for any API calls).

📄 Look at the error message carefully - it often tells you exactly what's wrong.


## Task 1: Start Your Notebook and Enable Model Access

In Terminal or command prompt window, navigate to the folder created in the previous step:

```bash
cd exercise4/
```

Then, run: `jupyter notebook`

This will open a Jupyter Notebook in a browser.

1. Choose **New**
2. Select **Python 3**. This will open a new tab.
3. Change the file name from Untitled to `prompt_management` by clicking on the file name next to the jupyter logo.
4. Select **Save Notebook**.

Now you will enable model access in Amazon Bedrock for the Amazon Nova Micro model.

1. Open the AWS Management Console and sign into your AWS account.
2. Navigate to the Amazon Bedrock dashboard by typing **Bedrock** into the search bar.
3. In the navigation pane, select **Model access**.
4. Select the **Modify model access** button.
5. In the Base models search bar, type **Nova micro**.
6. Check the check-box for the Amazon Nova Micro model.
7. Scroll down, and select **Next**.
8. On the next screen, select **Submit**.

✏️ It may take several minutes to receive access to the model. Refresh the page until it says **Access granted** in the Access status column for the Amazon Nova Micro Model. Then proceed to the next task.


## Task 2: Initialize Bedrock and Set Defaults

You will be utilizing multiple clients when working with Amazon Bedrock. Each of the following boto3 clients serves a specific purpose.

✏️ Amazon Bedrock provides the following service endpoints:

- **bedrock** – Contains control plane APIs for managing, training, and deploying models.
- **bedrock-runtime** – Contains data plane APIs for making inference requests for models hosted in Amazon Bedrock.
- **bedrock-agent** – Contains control plane APIs for creating and managing agents, knowledge bases, prompt management, and prompt flows.
- **bedrock-agent-runtime** – Contains data plane APIs for invoking agents and flows, and querying knowledge bases.

Reference the [Amazon Bedrock endpoints documentation](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_Operations_Amazon_Bedrock.html) to learn more about each service endpoint and what actions they support.

You will be using Amazon Nova Micro for the model for this exercise. Reference the [Amazon Nova Micro, Lite, Pro, and Premier AWS AI Service Card](https://aws.amazon.com/aig/service-cards/amazon-nova-micro-lite-pro-premier/) to learn more about this model.

In the first cell, in the Jupyter notebook, copy and paste the following code. Then, run the cell:


In [1]:
import boto3
import json
import datetime
from IPython.display import display, JSON

## Task 3: Create a Prompt Template

Amazon Bedrock makes it easy to create reusable prompt templates through Prompt Management. Instead of writing the same prompt over and over in your code, you can save a template once and use it across different workflows and plug in variables as needed.

When you create a prompt, you choose the model you want to run it with and customize how that model behaves using inference settings. You can even test different versions (called "variants") of your prompt to see which one gives you the best results.

✏️ As you refine your prompt, you can save versions along the way so you can experiment without losing your progress.

When you're ready, you can integrate the prompt into your app by referencing it directly in model inference calls or by adding it to a Bedrock Flow.

You'll create a basic prompt template with variables for an HR system that given a context information about a specific job opening will generate a job description.

To create a new prompt template using the `create_prompt` API, copy the following code:


✅ **Expected output:** A JSON response object containing information about the created prompt. A prompt ARN.

Run the code and explore the output from Amazon Bedrock. There is a printed JSON object that you can expand to learn more about what type of information Bedrock returns when a prompt template is created, and there is a prompt ARN which will be used in later steps.


In [2]:


bedrock_agent = boto3.client("bedrock-agent")
bedrock_runtime = boto3.client("bedrock-runtime")
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime")

model_id = "amazon.nova-micro-v1:0"

## Task 4: Create a New Version of the Prompt

When you save a prompt, it starts out as a draft. You can keep updating and tweaking that draft. Whether it's the wording, variables, or configuration, you can modify and update the prompt until you're happy with it.

Once you're ready to use the prompt in a real application, [you can create a version of it](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-create-version.html).

✏️ A version captures exactly how your prompt looked at that moment. You should create a version when you're confident in the prompt and want to lock it in for production use.

Having versions makes it easy to manage changes. You can switch between different prompt versions, test variations, and update your app with the one that fits your use case best.

Create a new prompt version by running the following code.


✅ **Expected output:** A prompt ARN with a `:X` at the end of it which defines the version number.


## Task 5: Invoke a Model Using the Versioned Prompt Template

To invoke the model using a prompt created using Prompt Management you need to use the `converse` API as documented in the [boto3 documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-runtime/client/converse.html), not the direct `invoke_model` API.

✏️ The Converse API in Amazon Bedrock is a high-level API that lets you invoke a model using resources like:

- Prompts (created and stored in Prompt Management)
- Knowledge Bases
- Agents
- Tools

Converse provides a consistent interface that works with all models that support messages. This allows you to write code once and use it with different models. If a model has unique inference parameters, you can also pass those unique parameters to the model.

Rather than sending a raw prompt string every time (as you would with the InvokeModel API), the Converse API allows you to pass in a `promptId` and supply the input variables.

You will learn more about the Converse API in the video lessons.

To invoke the model using the converse API, copy the following code:


✅ **Expected output:** Output from the model defining the job description based off of the input.


## Task 6: Optimize the Prompt

Amazon Bedrock offers a tool to [optimize prompts](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-optimize.html) that you can access via the AWS Management Console or the `bedrock-agent-runtime` service endpoint.

Optimization rewrites prompts to yield inference results that are more suitable for your use case. You can choose the model that you want to optimize the prompt for and then generate a revised prompt.

After you submit a prompt to optimize, Amazon Bedrock analyzes the components of the prompt. If the analysis is successful, it then rewrites the prompt. You can then copy and use the text of the optimized prompt and create a new variant for the prompt.

Use the `optimize_prompt` API to have Amazon Bedrock analyze the prompt for the target model.


✏️ There is a function defined to handle the response stream coming back from the call to the `optimize_prompt` API.

There is a variable called `prompt_input` that sets up the prompt template text bedrock will analyze. This is referencing a `template_text` variable created in an earlier step.

Then there is the call to the `optimize_prompt` API using the `bedrock_agent_runtime` client.

✅ **Expected output** should be similar to the following:

The optimized prompt will be formatted in Markdown with clear structure and guidelines.

✏️ The `optimize_prompt` API is designed to return prompts that are formatted in a way that's optimized for the target model, and many models work well with Markdown formatting for structured content.

The API is returning the optimized prompt with Markdown formatting because it's determined that this structure (with headers, bullet points, etc.) is more effective for the target model:

- Organizing the prompt into clear sections
- Providing better structure for the model to understand the task
- Improving the model's ability to follow instructions

Compare and contrast the two different versions of the prompt.

In the next task, you will use this new optimized prompt to create a new version of the prompt.


## Task 7: Update Prompt and Create New Prompt Version

In this task, you will update the prompt based on the output from the optimization step and create a new version.

To update the prompt template with the new optimized prompt, copy the following code:

✏️ The code first retrieves the latest draft version of the prompt using the `get_prompt` API.

It updates the text of the TEXT variant with your optimized content.

Then it calls `update_prompt` to save the changes.

Finally, it retrieves and prints the updated version to confirm the update.

**Note:** In the code below, you'll need to replace `<FILL_ME_IN_1>` with the ARN of the original prompt (from Task 3, not the versioned ARN). Replace `<FILL_ME_IN_2>` with the optimized prompt text from the previous step.


Next, create a new version of the prompt.

✅ **Expected output:** A prompt ARN with a `:X` at the end, indicating the version number.


## Task 8: Compare Versions

In this task, you will first invoke the model using the original prompt. Then, you will invoke the model using the updated prompt and compare the two outputs.

✏️ Using prompt version specific identifiers, you can more easily experiment with multiple versions of prompts, and you can externalize prompt identifiers into configuration files or environment variables. This makes it so that you can update your prompts without needing to deploy your full code and application.

**Note:** In the code below, replace `<FILL_ME_IN>` with the ARN for the first version of the prompt that was created in task 4. Ensure both of the `< >` characters are removed and only the prompt ARN in double quotes is present.


Now you will run a test using the same input parameter values, but this time using the optimized prompt.

**Note:** Replace `<FILL_ME_IN>` with the ARN for the second version of the prompt that was created in task 7. Ensure both of the `< >` characters are removed and only the prompt ARN in double quotes is present.

Compare the differences between the output for the original prompt and the output for the optimized prompt.


## Task 9: Clean Up

You will now clean up any resources you created during this exercise.

Run the following code if you want to delete all prompts and versions in your AWS account.

⚠️🚨⚠️ **This will delete any prompt existing in your account in the configured region, not just prompts created in this exercise.** ⚠️🚨⚠️

To delete all prompts in your AWS account, copy and paste the following code into a new cell. Then, run the code:


✅ **Expected output:**

```
Deleting prompt: job-description-1231342342341231 (ID: XXXXXXXXXX)
Deleting prompt: job-description-1231232131243132 (ID: XXXXXXXXXX)
All prompts deleted successfully
```

❌ If at any point the code for deleting resources fails, or if you want to delete only specific prompts, you can use the AWS Management Console to delete created resources.

To use the AWS Management Console to delete any prompts created with Prompt Management:

1. Navigate to the Amazon Bedrock by typing **bedrock** into the search bar.
2. Select **Prompt Management** from the navigation pane.
3. Select the prompt to delete.
4. Select **Delete**.
5. Repeat steps 3-4 for all prompts you wish to delete.

## Exercise Complete

You now know how to:

✅ Create and invoke reusable prompts

✅ Optimize prompt templates

✅ Iterate with improved versions

✅ Inspect and clean up resources


In [ ]:
prompt_name = f"job-description-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

template_text = '''
You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words
'''
response = bedrock_agent.create_prompt(
    name=prompt_name,
    description="Generates inclusive job descriptions from structured inputs",
    defaultVariant="v1",
    variants=[
        {
            "name": "v1",
            "modelId": model_id,
            "templateType": "TEXT",
            "templateConfiguration": {
                "text": {
                    "inputVariables": [
                        {"name": "job_title"},
                        {"name": "responsibilities"},
                        {"name": "requirements"},
                        {"name": "location"},
                        {"name": "work_type"}
                    ],
                    "text": template_text
                }
            },
            "inferenceConfiguration": {
                "text": {
                    "maxTokens": 500,
                    "temperature": 0.7,
                    "topP": 0.9,
                    "stopSequences": []
                }
            }
        }
    ]
)
print("\n==================== Response Object ====================\n")
display(JSON(response))

print("\n==================== Prompt ARN ====================\n")
prompt_arn = response['arn']
print(prompt_arn)


==================== Response Object ====================



<IPython.core.display.JSON object>


==================== Prompt ARN ====================

arn:aws:bedrock:us-east-1:458806987020:prompt/BHX6FCX97R


In [8]:
response = bedrock_agent.create_prompt_version(
    description='Initial prompt for creating job description documents.',
    promptIdentifier=prompt_arn
)

print("\n==================== PROMPT VERSION ARN ====================\n")
new_prompt_arn = response["arn"]
print(new_prompt_arn)


==================== PROMPT VERSION ARN ====================

arn:aws:bedrock:us-east-1:458806987020:prompt/BHX6FCX97R:1


In [9]:
prompt_arn_with_version = new_prompt_arn

response = bedrock_runtime.converse(
    modelId=prompt_arn_with_version,
    promptVariables={
        "job_title": {"text": "UX Designer"},
        "responsibilities": {"text": "Design user interfaces, run usability testing, collaborate with product teams"},
        "requirements": {"text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"},
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"}
    }
)

print("\n==================== Response Text ====================\n")
print(response['output']['message']['content'][0]['text'])


==================== Response Text ====================

**Job Title: UX Designer**

**Summary:**
We are seeking a talented and inclusive UX Designer to join our dynamic team. As a UX Designer, you will play a pivotal role in designing intuitive user interfaces, conducting usability testing, and collaborating closely with our product teams to deliver exceptional user experiences. This full-time position offers flexibility with options to work remotely or in our New York office.

**Responsibilities:**
- Design compelling and user-friendly interfaces that enhance user engagement.
- Conduct thorough usability testing to ensure optimal user experience.
- Collaborate effectively with cross-functional product teams to align design with business goals.
- Utilize your expertise in Figma, HTML, and CSS to bring your designs to life.

**Requirements:**
- 3+ years of experience in UX design.
- Proficiency in Figma for design prototyping.
- Strong knowledge of HTML and CSS.
- Excellent communicat

In [10]:
def handle_response_stream(response):
    try:
        event_stream = response['optimizedPrompt']
        for event in event_stream:
            if 'optimizedPromptEvent' in event:
                print("\n==================== OPTIMIZED PROMPT ====================\n")
                print(event['optimizedPromptEvent']['optimizedPrompt']['textPrompt']['text'])
    except Exception as e:
        raise e


prompt_input = {
    "textPrompt": {
        "text": template_text
    }
}

response = bedrock_agent_runtime.optimize_prompt(
            input=prompt_input,
            targetModelId=model_id
        )

print("\n==================== ORIGINAL PROMPT ====================\n")
print(template_text)
handle_response_stream(response)


==================== ORIGINAL PROMPT ====================


You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words


==================== OPTIMIZED PROMPT ====================

"# HR Job Description Generator\n\n## Task\nCreate a professional, inclusive job description based on the provided information.\n\n## Input Information\n### Job Title\n{{job_title}}\n\n### Responsibilities\n{{responsibilities}}\n\n### Requirements\n{{requirements}}\n\n### Location\n{{location}}\n\n### Work Type\n{{work_type}}\n\n## Guidelines\n1. Begin with a clear, compelling summary of the position (1-2 sentences)\n2. Structure the job description with clear headings:\n   - About the Role\n   - Key Responsibilities\n   - Qualificatio

In [11]:
prompt_identifier = prompt_arn

existing_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_identifier)
print("\n==================== ORIGINAL PROMPT ====================\n")
print(existing_prompt["variants"][0]["templateConfiguration"]["text"]["text"])


updated_variants = existing_prompt["variants"]
for variant in updated_variants:
    if variant["templateType"] == "TEXT":
        variant["templateConfiguration"]["text"]["text"] = new_prompt_arn


response = bedrock_agent.update_prompt(
    promptIdentifier=prompt_identifier,
    name=existing_prompt["name"],
    description=existing_prompt["description"],
    defaultVariant=existing_prompt["defaultVariant"],
    variants=updated_variants
)

print("\n==================== UPDATED PROMPT ====================\n")

updated_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_identifier)
print(updated_prompt["variants"][0]["templateConfiguration"]["text"]["text"])


==================== ORIGINAL PROMPT ====================


You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words


==================== UPDATED PROMPT ====================

arn:aws:bedrock:us-east-1:458806987020:prompt/BHX6FCX97R:1


In [ ]:
prompt_arn_ = prompt_arn

response = bedrock_runtime.converse(
    modelId=prompt_arn_,
    promptVariables={
        "job_title": {"text": "UX Designer"},
        "responsibilities": {"text": "Design user interfaces, run usability testing, collaborate with product teams"},
        "requirements": {"text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"},
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"}
    }
)

print("\\n==================== Response Text ====================\\n")
print(response['output']['message']['content'][0]['text'])

\n==================== Response Text ====================\n
It looks like you've provided an Amazon Web Services (AWS) resource identifier, specifically for a prompt within the Bedrock service in the us-east-1 region. Here's a breakdown of the identifier:

- `arn:aws:bedrock:us-east-1:458806987020:prompt/BHX6FCX97R:1`

- `arn`: This is the Amazon Resource Name (ARN) prefix, which is used to uniquely identify AWS resources.

- `aws`: This indicates the service provider, which in this case is AWS.

- `bedrock`: This specifies the service, which is Bedrock in this context. Bedrock is a service that provides fully managed machine learning models.

- `us-east-1`: This is the region code, indicating that the resource is located in the US East (N. Virginia) region.

- `458806987020`: This is the AWS account ID of the owner of the resource.

- `prompt/BHX6FCX97R:1`: This is the specific identifier for the prompt within the Bedrock service. The `BHX6FCX97R` part likely refers to a unique name o

In [13]:
new_prompt_arn_ = new_prompt_arn

response = bedrock_runtime.converse(
    modelId=new_prompt_arn_,
    promptVariables={
        "job_title": {"text": "UX Designer"},
        "responsibilities": {"text": "Design user interfaces, run usability testing, collaborate with product teams"},
        "requirements": {"text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"},
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"}
    }
)

print("\\n==================== Response Text ====================\\n")
print(response['output']['message']['content'][0]['text'])

\n==================== Response Text ====================\n
**Job Title: UX Designer**

**Summary:**
We are seeking a talented UX Designer to join our innovative team in New York or remotely. As a UX Designer, you will play a pivotal role in designing intuitive user interfaces, conducting usability testing, and collaborating closely with our diverse product teams to enhance user experience.

**Responsibilities:**
- Design compelling and user-friendly interfaces.
- Conduct thorough usability testing to refine and improve user interactions.
- Collaborate with cross-functional teams to ensure seamless product development.
- Utilize Figma for prototyping and design.
- Implement HTML/CSS to bring designs to life.

**Requirements:**
- 3+ years of experience in UX design.
- Proficiency in Figma, HTML, and CSS.
- Strong communication skills to articulate design rationale and collaborate effectively.
- Ability to work in a dynamic, inclusive environment.

**Location:** New York or remote

**Wor

In [14]:
response = bedrock_agent.list_prompts()

for prompt in response['promptSummaries']:
    prompt_id = prompt['id']  # The correct key is 'id', not 'promptId'
    print(f"Deleting prompt: {prompt['name']} (ID: {prompt_id})")
    bedrock_agent.delete_prompt(promptIdentifier=prompt_id)

print("All prompts deleted successfully")

Deleting prompt: job-description-20251105210105 (ID: BHX6FCX97R)
All prompts deleted successfully
